In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#import os
#os.getcwd()

In [ ]:
data_path = None #Add your path here
data_ip = pd.read_csv(data_path+r'\data_ip.csv',index_col=0)

In [ ]:
data_ip.head()

In [ ]:
data_ip.shape

In [ ]:
data_ip.describe()

In [ ]:
data_ip.isna().sum(axis=0)

In [ ]:
data_ip.dtypes

In [ ]:
data_ip[['ice_on','ice_off']] = data_ip[['ice_on','ice_off']].apply(pd.to_datetime)

In [ ]:
data_ip.dtypes

In [ ]:
data_ip.head()

In [ ]:
data_ip['ice_on_doy'] = np.where(data_ip.ice_on.dt.year < data_ip.year,
                                        data_ip.ice_on.dt.day_of_year,
                                        data_ip.ice_on.dt.day_of_year + 365)
data_ip['ice_off_doy'] = np.where(data_ip.ice_off.dt.year == data_ip.year,
                                        data_ip.ice_off.dt.day_of_year,
                                        data_ip.ice_off.dt.day_of_year - 365)

In [ ]:
data_ip.describe()

Here we plot day-of-year distribution for the ice-on and ice-off dates. In previous code we fixed the days a bit. If the ice-on day is delayed to next years side, we added 365 days to this observation. If the ice-off day occurs already in previous year, we subtracted 365 from this observation. So, the ice-on-day can be above 365 and the ice-off day can be minus signed.

In [ ]:
plt.style.use('dark_background')
plt.grid(linestyle='--',alpha=0.5)
data_ip.ice_off_doy.hist(color='cyan',
                         bins=75,
                         edgecolor='black')
plt.title('Ice-off day-of-year distribution',
          fontsize=18,
          color='lime')
plt.ylabel('Observation amount')
plt.xlabel('day-of-year')
#plt.savefig('ice-off_day-of-year.png',bbox_inches='tight')
plt.show()

In [ ]:
plt.grid(linestyle='--',alpha=0.5)
data_ip.ice_on_doy.hist(color='yellow',
                        bins=75,
                        edgecolor='black')
plt.title('Ice-on day-of-year distribution',
          fontsize=18,
          color='lime')
plt.ylabel('Observation amount')
plt.xlabel('day-of-year')
#plt.savefig('ice-on_day-of-year.png',bbox_inches='tight')
plt.show()

In [ ]:
data_ip['ice_cover_duration'] = (data_ip.ice_off-data_ip.ice_on).dt.days

In [ ]:
data_ip[(data_ip.ice_cover_duration > 365) | (data_ip.ice_cover_duration < 0)]

In [ ]:
duration_table = data_ip[(data_ip.ice_cover_duration <= 365) & (data_ip.ice_cover_duration >= 0)].copy()

In [ ]:
#data_ip.describe()

In [ ]:
#duration_table.describe()

In [ ]:
plt.grid(linestyle='--',alpha=0.5)
duration_table.ice_cover_duration.hist(color='red',
                                       bins=75,
                                       edgecolor='black')
plt.title('Ice-cover duration distribution',
          fontsize=18,
          color='lime')
plt.ylabel('Observation amount')
plt.xlabel('Duration as days')
#plt.savefig('ice_cover_duration.png',bbox_inches='tight')
plt.show()

Fitting polynomial model to all observations with Scikit-Learn and Statsmodels:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

polynomial_model = make_pipeline(PolynomialFeatures(2),
                                 LinearRegression())
polynomial_model.fit(duration_table[['year']],duration_table[['ice_cover_duration']])

min_year = duration_table[['year']].min().item()
max_year = duration_table[['year']].max().item()

years = [i for i in range(min_year,max_year+1)]
duration_preds =polynomial_model.predict(pd.DataFrame(data= years,columns=['year']))


In [ ]:
plt.style.use('dark_background')
plt.scatter(duration_table[['year']],duration_table[['ice_cover_duration']],marker='.')
plt.plot(years,duration_preds,color='red')
plt.show()

In [ ]:
polynomial_model.steps[1][1].coef_,polynomial_model.steps[1][1].intercept_

In [ ]:
import statsmodels.formula.api as smf

poly_model_duration = smf.ols("ice_cover_duration ~ year + I(year**2)", data=duration_table).fit()

In [ ]:
print(poly_model_duration.summary())

Fitting model to annual averages:

In [ ]:
annual_means = duration_table.groupby(['year'])[['ice_cover_duration']].mean().reset_index()
#annual_means

In [ ]:
import statsmodels.formula.api as smf

annual_model_duration = smf.ols("ice_cover_duration ~ year + I(year**2)", data=annual_means).fit()

In [ ]:
print(annual_model_duration.summary())

In [ ]:
annual_preds =annual_model_duration.predict(annual_means[['year']])

In [ ]:
plt.style.use('dark_background')
plt.grid(linestyle='--',alpha=0.5)
plt.plot(annual_means.year,annual_means.ice_cover_duration,color='yellow')
plt.plot(annual_means.year,annual_preds,color='red')
plt.show()


## Here we start to handle ltbl_ice data:

In [ ]:
data_path = None #Add your path here
ltbl_ice = pd.read_csv(data_path+r'\ltbl_ice.csv',index_col=0)

In [ ]:
ltbl_ice

In [ ]:
ltbl_ice.nlargest(10,'lat_wgs84')

In [ ]:
ltbl_ice.nsmallest(10,'lat_wgs84')

## Trying to figure out the coordinates:

In [ ]:
np.unique(ltbl_ice.groupby('lake_id')['lat_wgs84'].count(),return_counts=True)

In [ ]:
np.unique(ltbl_ice.groupby('station_id')['lat_wgs84'].count(),return_counts=True)

In [ ]:
lake_stations = ltbl_ice.groupby('lake_id')['station_id'].unique()
lake_stations

In [ ]:
lake_longitudes = ltbl_ice.groupby('lake_id')['lon_wgs84'].unique()
lake_longitudes

In [ ]:
lakes_with_many_stations =  pd.DataFrame(lake_stations)
lakes_with_many_stations = lakes_with_many_stations[lakes_with_many_stations.iloc[:,0].apply(lambda x: len(x)>1)]
lakes_with_many_stations.head()

In [ ]:
lmws_ids = lakes_with_many_stations.index.to_list()

Here we choose a subset of 'ltbl_ice' dataset. We pick only lakes, which have several station id's. Based on this data frame, columns 'cent_lat_wgs84' and 'cent_lon_wgs84' seem to remain constants for each lake, and columns 'lat_wgs84' and 'lon_wgs_84' are varying. This would most likely mean that the latter ones are linked to station location.

In [ ]:
pd.set_option('display.max_rows',150)
ltbl_ice[ltbl_ice['lake_id'].isin(lmws_ids)].sort_values('lake_id').reset_index(drop=True).head(25)

Here we spot some stations sharing the same location. Perhaps these cases are just recorded with different logic than the others? Maybe some station has changed its name/id or they are just reporting constant location to all stations.

In [ ]:
no_match_longitudes = {}
no_match_stations = {}
for i in range(len(lake_longitudes)):
    if ((lake_longitudes.index[i] != lake_stations.index[i])):
        print(f'Station and location indexes (lake_id) did not match for index {i}')
    if (len(lake_longitudes.iloc[i]) != len(lake_stations.iloc[i])) and ((lake_longitudes.index[i] == lake_stations.index[i])):
        print(f'Different amount of stations and longitudes in index: {i}')
        print(f'longitude index: {lake_longitudes.index[i]}')
        print(f'longitudes: {lake_longitudes.iloc[i]}')
        print(f'station index: {lake_stations.index[i]}')
        print(f'station: {lake_stations.iloc[i]}')
        print()
        no_match_longitudes[i] = lake_longitudes.iloc[i]
        no_match_stations[i] = lake_stations.iloc[i]

In [ ]:
no_match_longitudes,no_match_stations

In [ ]:
list(no_match_stations.values())

## Some findings this far:

* Most likely 'latwgs84' and 'lon_wgs84' are ment to be coordinates for stations. Columns 'cent_lat_wgs84' and 'cent_lon_wgs84' look like coordinates for lakes.

* Based on previous testing and data frame below, there are 4 lakes in this subset for which we find several station_id's, but the 'lat_wgs84' and 'lon_wgs84' are the same.

Lets make table for lakes, which have several station id's with same 'latwgs84' and 'lon_wgs84'. These observations are from lakes in Canada.

In [ ]:
trouble_lakes = [lake for lakes in list(no_match_stations.values()) for lake in lakes]
df_trouble_lakes = ltbl_ice[ltbl_ice['station_id'].isin(trouble_lakes)]
df_trouble_lakes

Next we continue data handling.
* Here we remove 4 columns from ltbl_ice data. Keeping the location columns, which appears to show the station location.
* Also creating new table where we filter out all empty values in max depth column.

In [ ]:
ltbl_ice.drop(columns=['subset','depth_mean_m','cent_lat_wgs84','cent_lon_wgs84'],inplace=True)

In [ ]:
ltbl_ice_depth = ltbl_ice[~ltbl_ice.depth_max_m.isna()]
ltbl_ice_depth.reset_index(drop=True,inplace=True)

In [ ]:
ltbl_ice_depth.shape

* Distribution of stations by country:

In [ ]:
np.unique(ltbl_ice_depth.country,return_counts=True)

Creating dt_ice kind of table with left-join. Trying to achieve similar table as Joachim created, but with more narrowed and clean observations. We have
* Dropped all lakes with no max-depth information in ltbl_ice.csv based data.
* Removed columns ['subset','depth_mean_m','cent_lat_wgs84','cent_lon_wgs84'] from ltbl_ice.csv based data

Next steps:
* Below we left-join the cleaned ltbl_ice named as 'ltbl_ice_depth' to data_ip.csv based data frame
* After this, we drop all rows where we do not have more detailed lake-data. In these rows we have observation in data_ip which does not match any lake information in ltbl_ice.

In [ ]:
dt_ice_depth = pd.merge(data_ip,ltbl_ice_depth,how='left',on='station_id')

In [ ]:
#dt_ice_depth

In [ ]:
#dt_ice_depth.describe()

Checkin the size of frame, if we drop some NaN-rows:

In [ ]:
#dt_ice_depth.dropna(axis=0).shape

In [ ]:
#dt_ice_depth.isna().sum(axis=0)

We have stations that did collect only one end of the observation. Both observations are never missing at the same time. ice_on NaN's total 4834 and ice_off 1613 => Sum of this 4834+1613=6447 is exact match with the amount of missing ice_over_duration values.

* Below we first drop the rows, which do not have more detailed knowledge about the lake.
* Then doing some analysis for different kinds of groups in this data.

In [ ]:
df_depth_one_obs = dt_ice_depth[~dt_ice_depth.lake_id.isna()].copy()
df_depth_one_obs.reset_index(drop=True,inplace=True)

In [ ]:
#df_depth_one_obs

In [ ]:
#df_depth_one_obs.describe()

Here we analyse the annual distibution of latitude information in 'lat_wgs84' column:

In [ ]:
year_lats = df_depth_one_obs.groupby(by='year')['lat_wgs84']

Here is a function to create statistics table. Most of the stats are already under .describe() method, but median is missing. This helps choosing subset of .describe() table and optionally adding median to group of statistics.

In [ ]:
def create_group_stats_table(group_object,stats):
    ''' 
    Function takes pandas.core.groupby.generic.SeriesGroupBy object as input,
    and returns data frame of chosen statistics for this group-object.
    Median is added if chosen in stats, otherwise using just stats under .describe() method.'''
    
    des_cols = [x for x in stats if x in group_object.describe().columns.to_list()]
    df = group_object.describe()[des_cols]
    if 'median' in stats:
        df = pd.concat([pd.DataFrame(data=group_object.median().to_numpy(),
                        index=df.index,
                        columns=[
                           'median',
                           ]),
                           df],
                        axis=1)
    return df

In [ ]:
df_lats = create_group_stats_table(group_object=year_lats,
                                   stats=['mean','median','25%','75%','min','max','std'])
#pd.set_option('display.max_rows',100)
#df_lats

Here we create function to visualize the statistics table:

In [ ]:
def visualize_stats(df,
                    title,
                    titlecolor,
                    linecolors,
                    figsize,
                    xlabel,
                    ylabel,
                    ylimits=None,
                    file_name=None):
    
    '''
    Function takes statistics data frame as input. Create frame with function create_group_stats_table, or some similar method.
    Parameter names are closely related to needed matplotlib parameters.
    df: Pandas data frame.
    title: text for title.
    titlecolor: color for title.
    linecolors: colors for each statistic. A list of colors used with matplotlib.
    figsize: figsize.
    xlabel: xlabel.
    ylabel: ylabel.
    ylimits: optional. Determines ylim -parameter. x-axis has the grouping range automatically as limits.
    filename: optional. If you want to save the figure, give the filename and it will be saved to current directory.
    '''
    df.plot(color=linecolors,
                  figsize=figsize)
    plt.style.use('dark_background')
    plt.grid(linestyle='--',alpha=0.5)
    plt.title(title,
              color=titlecolor,
              fontsize=18)
    
    if ylimits is not None:
        plt.ylim(ylimits)
        
    plt.ylabel(ylabel=ylabel,fontsize=14)
    plt.xlabel(xlabel=xlabel,fontsize=14)

    legend = plt.legend(title='Statistics',
              fontsize=13,
              bbox_to_anchor=(1,1),
              loc='upper left')
    
    legend.get_title().set_fontsize(13)
    if file_name is not None:
        plt.savefig(file_name,
                    bbox_inches='tight')
    plt.show()

    


We removed redundant columns and then drop out all rows including NaNs in 'lake details' -columns. When visualizing just rows with somewhat full information, we see a significant decreasing in lower end of latitude distribution a few years ago. Unusually low 25% quantile is apparent for years 2017-2018 only.

* What are the observations, which do not have detailed lake information in last columns? All information about lakes, including the latitude information, is missing from these rows. Seems that there was no matching station_id in 'ltbl_ice.csv' for these observations, but the station exists in 'data_ip.csv'.

In [ ]:
colors = ['cyan','lime','yellow','red','blue','purple','white',]
visualize_stats(df=df_lats,
                title='Statistics of annual latitudes',
                titlecolor='yellow',
                linecolors=colors,
                figsize=(8,6),
                xlabel='Year',
                ylabel='Latitude',
                ylimits=(0,90),
                #file_name='annual_altitude_statistics.png'
                )


Some extra checks for latitude statistics:

In [ ]:
# year_lats.count().plot()
# plt.grid(linestyle='--',alpha=0.5)
# plt.show()

In [ ]:
# year_lats.std().plot(color='yellow')
# plt.grid(linestyle='--',alpha=0.5)
# plt.ylim((0,8))
# plt.show()

A guick look at the distribution of the latitude statistics during the years 1950-2024:

In [ ]:
plt.style.use('dark_background')
df_lats.hist(bins=15,figsize=(7,9),color='lime');

Repeating same analysis for longitudes:

In [ ]:
#df_depth_one_obs.head()

In [ ]:
year_longs = df_depth_one_obs.groupby(by='year')['lon_wgs84']

df_longs = create_group_stats_table(group_object=year_longs,
                                   stats=['mean','median','25%','75%','min','max','std'])


In [ ]:
# pd.set_option('display.max_rows',100)
# df_longs

In [ ]:
from random import sample,shuffle,seed
seed(27)
colors = np.array(['cyan','lime','yellow','red','blue','purple','white'])
visualize_stats(df=df_longs,
                title='Statistics of annual longitudes',
                titlecolor='yellow',
                # Randomly picking colors
                linecolors=colors[sample([i for i in range(len(colors))],len(colors))],
                figsize=(8,6),
                xlabel='Year',
                ylabel='Longitude',
                #file_name='annual_longitude_statistics.png'
                )

Then for lake area, for which we use also log-values:

In [ ]:
year_areas = df_depth_one_obs.groupby(by='year')['area_ha']

df_areas = create_group_stats_table(group_object=year_areas,
                                   stats=['mean','median','25%','75%','min','max','std'])

In [ ]:
seed(29)
visualize_stats(df=df_areas,
                title='Statistics of annual lake area',
                titlecolor='yellow',
                # Randomly picking colors
                linecolors=colors[sample([i for i in range(len(colors))],len(colors))],
                figsize=(8,6),
                xlabel='Year',
                ylabel='Area (ha)',
                #file_name='annual_area_statistics.png'
                )

Adding first log(area) column to dataframe. Then repeating the visualization for annual log(area)-values:

In [ ]:
df_depth_one_obs['log_area_ha'] = np.log(df_depth_one_obs['area_ha'])

year_log_areas = df_depth_one_obs.groupby(by='year')['log_area_ha']

df_log_areas = create_group_stats_table(group_object=year_log_areas,
                                   stats=['mean','median','25%','75%','min','max','std'])

In [ ]:
seed(29)
visualize_stats(df=df_log_areas,
                title='Statistics of annual lake log-area',
                titlecolor='lime',
                # Randomly picking colors
                linecolors=colors[sample([i for i in range(len(colors))],len(colors))],
                figsize=(8,6),
                xlabel='Year',
                ylabel='Area log(ha)',
                #file_name='annual_log_area_statistics.png'
                )

Last visualization for maximum depth:

In [ ]:
year_depths = df_depth_one_obs.groupby(by='year')['depth_max_m']

df_depths = create_group_stats_table(group_object=year_depths,
                                   stats=['mean','median','25%','75%','min','max','std'])

In [ ]:
seed(77)
visualize_stats(df=df_depths,
                title='Yearly statistics for lake max depths',
                titlecolor='lime',
                # Randomly picking colors
                linecolors=colors[sample([i for i in range(len(colors))],len(colors))],
                figsize=(8,6),
                xlabel='Year',
                ylabel='Maximum depth of lake',
                #file_name='annual_max_depth_statistics.png'
                )

In [ ]:
np.unique(df_depth_one_obs.country,return_counts=True)

In [ ]:
df_depth_one_obs.shape

In [ ]:
df_depth_one_obs.describe()

In [ ]:
num_cols = ['lat_wgs84','lon_wgs84','altitude_m','area_ha','depth_max_m']
df_nums_orig= df_depth_one_obs[num_cols+['country']].copy().dropna(axis=0)
df_nums = df_nums_orig[num_cols]

In [ ]:
df_nums_orig

In [ ]:
df_nums

In [ ]:
df_nums_scaled = (df_nums - df_nums.mean(axis=0))/df_nums.std(axis=0)

In [ ]:
df_nums_scaled.describe()

In [ ]:
#plt.style.use('ggplot')
#sns.pairplot(df_nums_scaled)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA()
pca.fit(df_nums_scaled)

In [ ]:
print(pca.explained_variance_ratio_)

In [ ]:
df_nums_pca = pca.transform(df_nums_scaled)

In [ ]:
df_nums_pca.shape,df_nums_orig.shape

In [ ]:
#df_nums_orig.head()

In [ ]:
df_pca_plot = pd.DataFrame(data=df_nums_pca,columns=['PC'+str(i) for i in range(1,df_nums_pca.shape[1]+1)])
df_pca_plot['country'] = df_nums_orig.country
df_pca_plot['country_int'],_ = pd.factorize(df_pca_plot['country'])
df_pca_plot

In [ ]:
#import matplotlib
#matplotlib.style.available

This plot will be saved in dark bacground mode, but the plot area will change to whitegrid. The style determined earlier does effect even though we change it here. Visualization is more clear when looking at the saved image ''2D_PCA.png'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
pca_colors = ['cyan','lime','purple','red','blue']
plt.style.use('seaborn-v0_8-whitegrid')
scatterplot = sns.scatterplot(x='PC1',
                              y='PC2',
                              hue='country',
                              data = df_pca_plot,
                              palette=pca_colors,
                              s=13)

legend = plt.legend(title='country',
                    fontsize=12,
                    markerscale=2)
legend.get_title().set_fontsize(13)
plt.title('2D PCA for lake data',
          fontsize=18,
          color='cyan')
plt.xlabel('PC1',
           fontsize=14,
           color='yellow')
plt.ylabel('PC2',
           fontsize=14,
           color='yellow')

plt.xticks(color='lime')
plt.yticks(color='lime')

#plt.savefig('2D_PCA.png',
#            bbox_inches='tight')

plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12,12))
axis = fig.add_subplot(111,projection='3d')

cat_countries = df_pca_plot.country.unique()
colors_3D = {country:pca_colors[i] for i,country in enumerate(cat_countries)}

for ctr in cat_countries:
    df_sub = df_pca_plot[df_pca_plot.country==ctr]
    axis.scatter(df_sub.iloc[:,0],
                df_sub.iloc[:,1],
                df_sub.iloc[:,2],
                c=[colors_3D[ctr]],
                label=ctr,
                marker=".")
axis.set_title('3D PCA for lake data',
               color='yellow',
               fontsize=24)
axis.set_xlabel('PC1',
                fontsize=14,
                color='purple')
axis.set_ylabel('PC2',
                fontsize=14,
                color='purple')
axis.set_zlabel('PC3',
                fontsize=14,
                color='purple')

legend = plt.legend(title='country',
                    fontsize=13,
                    markerscale=3)
legend.get_title().set_fontsize(14)

plt.tight_layout()

#plt.savefig('3D_PCA.png',
#            bbox_inches='tight')
plt.show()

In [ ]:
np.unique(df_pca_plot.country_int,return_counts=True),df_pca_plot.country_int.unique(),df_pca_plot.country.unique()

Observations with no connection to detailed lake data. This is the data not included in analysis above. What exactly are these stations with no match in the lake dataset?

In [ ]:
df_no_lake_id = dt_ice_depth[dt_ice_depth.lake_id.isna()].copy()

In [ ]:
df_no_lake_id

In [ ]:
df_no_lake_id.describe()